## 2. Data Loading & Preprocessing

In [1]:
import pandas as pd

# TRAIN_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/train.csv"
# TEST_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/phase_1_test.csv"

TRAIN_FILE = "data/train.csv"
TEST_FILE = "data/phase_1_test.csv"


df_train = pd.read_csv(TRAIN_FILE)
display(df_train.head())
print(f"✓ Loaded {len(df_train)} training samples")
print(f"Columns: {df_train.columns.tolist()}")
print(f"\nFirst sample:")
print(f"Question preview: {df_train['question'].iloc[0][:200]}...")
print(f"Answer: {df_train['answer'].iloc[0]}")
print(f"\nAnswer distribution:")
print(df_train['answer'].value_counts())

,ID,question,answer
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test use...,C2
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test use...,C1
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test use...,C2
3,ID_D6C9N2X295,Analyze the 5G wireless network drive-test use...,C2
4,ID_8JC15PNP3Q,Analyze the 5G wireless network drive-test use...,C5


✓ Loaded 2400 training samples
Columns: ['ID', 'question', 'answer']

First sample:
Question preview: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 pote...
Answer: C2

Answer distribution:
answer
C5    352
C7    349
C3    330
C2    320
C4    283
C8    277
C1    264
C6    225
Name: count, dtype: int64


## 📊 Enhanced Preprocessing with Type Safety & Feature Engineering

**Key Improvements:**
1. ✅ **Type casting** - Ensures all numeric fields are properly typed (not strings)
2. ✅ **Missing neighbor RSRP fields** - Now included in float casting
3. ✅ **Centralized field definitions** - Easy to maintain
4. ✅ **RCA-friendly features** - Computed automatically for better model learning

**Why This Matters:**
- Prevents "123.4" string vs 123.4 float bugs in calculations
- Ensures distance/ratio computations work correctly
- Makes feature engineering more reliable

In [2]:
import re
import json
import math
import pandas as pd
from statistics import mean, median, stdev
from typing import List, Dict, Any, Optional

# ============================================================================
# LABEL MAPPING
# ============================================================================
CAUSE_TO_NUM = {f"C{i}": i for i in range(1, 9)}
NUM_TO_CAUSE = {i: f"C{i}" for i in range(1, 9)}

# ============================================================================
# TYPE CONVERSION UTILITIES
# ============================================================================

def to_float(x) -> Optional[float]:
    """Safely convert to float, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_int(x) -> Optional[int]:
    """Safely convert to int, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return int(float(x))  # float first to handle "123.0" strings
    except (ValueError, TypeError):
        return None

def sanitize_question_text(q: str) -> str:
    """
    Convert literal '\\n' to real newlines.
    CRITICAL: Don't use unicode decoding or you corrupt '\\boxed'
    """
    if q is None:
        return ""
    return q.replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

# ============================================================================
# DOMAIN-SPECIFIC CALCULATIONS
# ============================================================================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance in meters between two lat/lon points"""
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    """
    Map beam scenario to vertical beamwidth:
    - DEFAULT/SCENARIO_1-5: 6 degrees
    - SCENARIO_6-11: 12 degrees
    - SCENARIO_12+: 25 degrees
    """
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6

    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6

    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    elif 6 <= k <= 11:
        return 12
    else:  # 12+
        return 25

def electronic_tilt_deg(digital_tilt) -> float:
    """
    Convert digital tilt to degrees:
    - 255 => 6 degrees (special case)
    - Otherwise: value is degrees
    """
    try:
        v = int(float(digital_tilt))
        return 6.0 if v == 255 else float(v)
    except (ValueError, TypeError):
        return 6.0  # Default fallback

print("✓ Type conversion and domain utilities loaded")

✓ Type conversion and domain utilities loaded


In [3]:
# ============================================================================
# TABLE PARSING
# ============================================================================

def parse_pipe_table(table_text: str) -> List[Dict[str, Any]]:
    """
    Parse pipe-delimited table into list of dicts.
    Handles '-', '—', and empty values as None.
    """
    if not table_text:
        return []

    lines = [ln.strip() for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if "|" in ln]

    if not lines:
        return []

    # Parse header
    header = [h.strip() for h in lines[0].split("|")]

    # Parse rows
    rows = []
    for ln in lines[1:]:
        parts = [p.strip() for p in ln.split("|")]

        # Normalize row length to match header
        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        elif len(parts) > len(header):
            parts = parts[:len(header)]

        # Build record
        rec = {}
        for k, v in zip(header, parts):
            rec[k] = None if v in ("", "-", "—", "–") else v

        rows.append(rec)

    return rows

print("✓ Table parsing loaded")

✓ Table parsing loaded


In [4]:
# ============================================================================
# COLUMN MAPPING & TYPE CASTING (IMPROVED)
# ============================================================================

# Drive-test data column mappings
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",

    # Neighbor PCIs
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",

    # Neighbor RSRPs (CRITICAL - these were missing!)
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

# Engineering parameters column mappings
ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

# Define fields by type for proper casting
FLOAT_FIELDS = [
    "longitude", "latitude", "gps_speed_kmh",
    "ss_rsrp_dbm", "ss_sinr_db", "throughput_mbps", "dl_rb_num",
    "mechanical_downtilt", "digital_tilt", "height", "max_tx_power",
    "mechanical_azimuth", "digital_azimuth",
    # CRITICAL: Add neighbor RSRP fields
    "nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm",
    "nei4_rsrp_dbm", "nei5_rsrp_dbm",
]

INT_FIELDS = [
    "pci", "serving_pci",
    "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci",
]

def normalize_rows(rows: List[Dict], mapping: Dict[str, str]) -> List[Dict]:
    """
    Normalize column names and apply type casting.
    This ensures all numeric operations work correctly.
    """
    normalized = []

    for r in rows:
        nr = {}

        # Map column names
        for k, v in r.items():
            normalized_key = mapping.get(k)
            if normalized_key is None:
                continue  # Skip unmapped columns
            nr[normalized_key] = v

        # Type casting - CRITICAL for numeric operations
        for field in FLOAT_FIELDS:
            if field in nr:
                nr[field] = to_float(nr[field])

        for field in INT_FIELDS:
            if field in nr:
                nr[field] = to_int(nr[field])

        normalized.append(nr)

    return normalized

print("✓ Column mappings and type casting configured")
print(f"  - Float fields: {len(FLOAT_FIELDS)}")
print(f"  - Int fields: {len(INT_FIELDS)}")
print(f"  - Drive-test columns mapped: {len(DRIVE_MAP)}")
print(f"  - Engineering columns mapped: {len(ENG_MAP)}")

✓ Column mappings and type casting configured
  - Float fields: 18
  - Int fields: 7
  - Drive-test columns mapped: 19
  - Engineering columns mapped: 14


In [5]:
# ============================================================================
# RCA-FRIENDLY FEATURE ENGINEERING (STREAMLINED)
# ============================================================================

from typing import List, Dict, Any
import math
from statistics import mean, median, stdev

def compute_rca_features(drive_rows: List[Dict], eng_rows: List[Dict]) -> Dict[str, Any]:
    """
    Compute ONLY the features required by format_features_text.
    Optimized for efficiency - removed all unused calculations.
    
    Returns dict with 48 unique features used in the prompt.
    """
    features = {}
    
    # Helper: dBm -> mW
    def _dbm_to_mw(dbm_val: float) -> float:
        return 10 ** (dbm_val / 10.0)
    
    # Helper: safe correlation calculation
    def _safe_corr(xs, ys):
        xs2, ys2 = [], []
        for x, y in zip(xs, ys):
            if x is None or y is None:
                continue
            xs2.append(float(x))
            ys2.append(float(y))
        if len(xs2) < 2:
            return None
        mx, my = mean(xs2), mean(ys2)
        num = sum((x - mx) * (y - my) for x, y in zip(xs2, ys2))
        denx = sum((x - mx) ** 2 for x in xs2)
        deny = sum((y - my) ** 2 for y in ys2)
        if denx <= 0 or deny <= 0:
            return None
        return num / (math.sqrt(denx) * math.sqrt(deny))
    
    # -------------------------------------------------------------------------
    # C1 FEATURES: Excessive Downtilt
    # -------------------------------------------------------------------------
    serving_rsrps = [r["ss_rsrp_dbm"] for r in drive_rows if r.get("ss_rsrp_dbm") is not None]
    sinrs = [r["ss_sinr_db"] for r in drive_rows if r.get("ss_sinr_db") is not None]
    
    # rsrp_min_dbm, rsrp_mean_dbm, rsrp_10th_percentile
    if serving_rsrps:
        features["rsrp_min_dbm"] = min(serving_rsrps)
        features["rsrp_mean_dbm"] = mean(serving_rsrps)
        if len(serving_rsrps) > 2:
            features["rsrp_10th_percentile"] = sorted(serving_rsrps)[int(0.1 * len(serving_rsrps))]
        else:
            features["rsrp_10th_percentile"] = min(serving_rsrps)
    else:
        features["rsrp_min_dbm"] = None
        features["rsrp_mean_dbm"] = None
        features["rsrp_10th_percentile"] = None
    
    # sinr_degradation_db, sinr_std_when_rsrp_stable
    if serving_rsrps and sinrs and features["rsrp_mean_dbm"] is not None:
        sinr_mean = mean(sinrs)
        expected_sinr = features["rsrp_mean_dbm"] + 100
        features["sinr_degradation_db"] = expected_sinr - sinr_mean
        
        rsrp_std = stdev(serving_rsrps) if len(serving_rsrps) > 1 else 0.0
        sinr_std = stdev(sinrs) if len(sinrs) > 1 else 0.0
        denom = abs(rsrp_std) + 1e-6
        features["sinr_std_when_rsrp_stable"] = sinr_std / denom
    else:
        features["sinr_degradation_db"] = None
        features["sinr_std_when_rsrp_stable"] = None
    
    # handover_rsrp_delta_mean (computed later in handover section)
    
    # -------------------------------------------------------------------------
    # C2 FEATURES: Overshoot
    # -------------------------------------------------------------------------
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]
    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}
    
    # dist_max_m, dist_p95_m, overshoot_flag
    distances = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)
        
        if not cell:
            continue
        
        if (r.get("latitude") is None or r.get("longitude") is None or
            cell.get("latitude") is None or cell.get("longitude") is None):
            continue
        
        dist = haversine_m(
            r["latitude"], r["longitude"],
            cell["latitude"], cell["longitude"]
        )
        distances.append(dist)
    
    if distances:
        features["dist_max_m"] = max(distances)
        features["dist_p95_m"] = sorted(distances)[int(0.95 * (len(distances) - 1))]
        features["overshoot_flag"] = features["dist_p95_m"] > 1000 if features["dist_p95_m"] else False
    else:
        features["dist_max_m"] = None
        features["dist_p95_m"] = None
        features["overshoot_flag"] = False
    
    # handover_count
    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1
    features["handover_count"] = handover_count
    
    # throughput_variance_across_cells, best_cell_throughput_gap (computed later in C3)
    
    # -------------------------------------------------------------------------
    # C3 FEATURES: Wrong Cell Selection
    # -------------------------------------------------------------------------
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]
    
    # tp_samples_below_600, tp_drop_ratio
    if tps:
        features["tp_samples_below_600"] = sum(1 for x in tps if x < 600)
        features["tp_drop_ratio"] = sum(1 for x in tps if x < 600) / len(tps)
    else:
        features["tp_samples_below_600"] = 0
        features["tp_drop_ratio"] = None
    
    # Handover TP deltas and cross-cell throughput comparison
    ho_tp_deltas = []
    ho_rsrp_deltas = []
    cell_throughputs = {}
    
    prev = None
    for r in drive_rows:
        # Track throughput by PCI
        pci = r.get("serving_pci")
        tp = r.get("throughput_mbps")
        if pci and tp:
            cell_throughputs.setdefault(pci, []).append(tp)
        
        # Handover deltas
        if prev is None:
            prev = r
            continue
        
        prev_pci = prev.get("serving_pci")
        curr_pci = r.get("serving_pci")
        
        if prev_pci is not None and curr_pci is not None and curr_pci != prev_pci:
            prev_tp = prev.get("throughput_mbps")
            curr_tp = r.get("throughput_mbps")
            prev_rsrp = prev.get("ss_rsrp_dbm")
            curr_rsrp = r.get("ss_rsrp_dbm")
            
            if prev_tp is not None and curr_tp is not None:
                ho_tp_deltas.append(curr_tp - prev_tp)
            if prev_rsrp is not None and curr_rsrp is not None:
                ho_rsrp_deltas.append(curr_rsrp - prev_rsrp)
        
        prev = r
    
    features["handover_tp_delta_mean"] = mean(ho_tp_deltas) if ho_tp_deltas else None
    features["handover_rsrp_delta_mean"] = mean(ho_rsrp_deltas) if ho_rsrp_deltas else None
    features["handover_improves_tp_flag"] = (features["handover_tp_delta_mean"] is not None and 
                                             features["handover_tp_delta_mean"] > 80)
    
    # avg_throughput_change_after_handover, throughput_improved_by_handover
    features["avg_throughput_change_after_handover"] = mean(ho_tp_deltas) if ho_tp_deltas else 0.0
    features["throughput_improved_by_handover"] = features["avg_throughput_change_after_handover"] > 50
    
    # throughput_variance_across_cells, best_cell_throughput_gap
    if len(cell_throughputs) >= 2:
        cell_avg_tp = {pci: mean(tps) for pci, tps in cell_throughputs.items()}
        
        if serving_pcis:
            from collections import Counter
            most_common_pci = Counter(serving_pcis).most_common(1)[0][0]
            current_avg = cell_avg_tp.get(most_common_pci, 0)
            other_avgs = [avg for pci, avg in cell_avg_tp.items() if pci != most_common_pci]
            best_alt = max(other_avgs) if other_avgs else 0
            features["best_cell_throughput_gap"] = best_alt - current_avg
        else:
            features["best_cell_throughput_gap"] = 0.0
        
        best_tp_pci = max(cell_avg_tp.items(), key=lambda x: x[1])
        worst_tp_pci = min(cell_avg_tp.items(), key=lambda x: x[1])
        features["throughput_variance_across_cells"] = best_tp_pci[1] - worst_tp_pci[1]
    else:
        features["best_cell_throughput_gap"] = 0.0
        features["throughput_variance_across_cells"] = None
    
    # -------------------------------------------------------------------------
    # C4 FEATURES: Overlapping Coverage
    # -------------------------------------------------------------------------
    pci_to_gnodeb = {}
    for cell in eng_rows:
        if cell.get("pci") is not None:
            pci_to_gnodeb[cell["pci"]] = cell.get("gnodeb_id")
    
    # Get serving cell's gNodeB
    serving_gnodeb = None
    if serving_pcis:
        serving_pci = serving_pcis[-1]
        serving_gnodeb = pci_to_gnodeb.get(serving_pci)
    
    # noncolocated_strong_neighbor_gnodeb_count_mean, noncolocated_strong_neighbor_gnodeb_count_max
    noncolocated_strong_counts = []
    coloc_strong = 0
    noncoloc_strong = 0
    top1_top2_gaps = []
    
    neighbor_rsrp_by_pci = {}
    
    for r in drive_rows:
        serv_pci = r.get("serving_pci")
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_pci is None or serv_rsrp is None:
            continue
        
        serv_gnb = pci_to_gnodeb.get(serv_pci)
        if serv_gnb is None:
            continue
        
        # Track strong non-colocated neighbors per sample
        strong_noncolocated_gnbs = set()
        nei_vals = []
        
        for i in range(1, 6):
            nei_pci = r.get(f"nei{i}_pci")
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            
            if nei_pci is None or nei_rsrp is None:
                continue
            
            # Track for neighbor analysis
            if nei_pci not in neighbor_rsrp_by_pci:
                neighbor_rsrp_by_pci[nei_pci] = []
            neighbor_rsrp_by_pci[nei_pci].append(nei_rsrp)
            nei_vals.append(nei_rsrp)
            
            # Strong neighbor: RSRP > -95 AND within 6 dB of serving
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                nei_gnb = pci_to_gnodeb.get(nei_pci)
                if nei_gnb is not None:
                    if nei_gnb != serv_gnb:
                        strong_noncolocated_gnbs.add(nei_gnb)
                        noncoloc_strong += 1
                    else:
                        coloc_strong += 1
        
        noncolocated_strong_counts.append(len(strong_noncolocated_gnbs))
        
        # top1_top2_gap_db_mean
        if len(nei_vals) >= 2:
            sorted_nei = sorted(nei_vals, reverse=True)
            top1, top2 = sorted_nei[0], sorted_nei[1]
            top1_top2_gaps.append(top1 - top2)
    
    features["noncolocated_strong_neighbor_gnodeb_count_mean"] = (mean(noncolocated_strong_counts) 
                                                                    if noncolocated_strong_counts else 0.0)
    features["noncolocated_strong_neighbor_gnodeb_count_max"] = (max(noncolocated_strong_counts) 
                                                                  if noncolocated_strong_counts else 0)
    features["top1_top2_gap_db_mean"] = mean(top1_top2_gaps) if top1_top2_gaps else None
    
    # strong_neighbor_noncolocated_share
    denom = coloc_strong + noncoloc_strong
    features["strong_neighbor_noncolocated_share"] = (noncoloc_strong / denom) if denom > 0 else 0.0
    
    # high_interference_power_ratio_flag
    ratios_db = []
    for r in drive_rows:
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_rsrp is None:
            continue
        S = _dbm_to_mw(serv_rsrp)
        I = 0.0
        for i in range(1, 6):
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            if nei_rsrp is None:
                continue
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                I += _dbm_to_mw(nei_rsrp)
        if S > 0 and I > 0:
            ratios_db.append(10.0 * math.log10((I + 1e-12) / (S + 1e-12)))
    
    if ratios_db:
        ratios_db_sorted = sorted(ratios_db)
        p90_idx = int(0.9 * (len(ratios_db_sorted) - 1))
        features["high_interference_power_ratio_flag"] = ratios_db_sorted[p90_idx] > -3.0
    else:
        features["high_interference_power_ratio_flag"] = False
    
    # rsrp_advantage_of_best_neighbor
    best_neighbor_rsrp = None
    if neighbor_rsrp_by_pci:
        best_pci = max(neighbor_rsrp_by_pci.items(), key=lambda x: mean(x[1]))
        best_neighbor_rsrp = mean(best_pci[1])
    
    serving_rsrp_avg = mean(serving_rsrps) if serving_rsrps else None
    if best_neighbor_rsrp is not None and serving_rsrp_avg is not None:
        features["rsrp_advantage_of_best_neighbor"] = best_neighbor_rsrp - serving_rsrp_avg
    else:
        features["rsrp_advantage_of_best_neighbor"] = None
    
    # -------------------------------------------------------------------------
    # C5 FEATURES: Ping-Pong Handover
    # -------------------------------------------------------------------------
    # ping_pong_handover_count, ping_pong_detected, frequent_handover_flag
    ping_pong_events = 0
    prev_pci = None
    prev_prev_pci = None
    
    for r in drive_rows:
        curr_pci = r.get("serving_pci")
        if prev_pci and prev_prev_pci and curr_pci == prev_prev_pci and curr_pci != prev_pci:
            ping_pong_events += 1
        prev_prev_pci = prev_pci
        prev_pci = curr_pci
    
    features["ping_pong_handover_count"] = ping_pong_events
    features["ping_pong_detected"] = ping_pong_events >= 2
    features["frequent_handover_flag"] = handover_count >= 3
    
    # Note: handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean already computed above
    
    # -------------------------------------------------------------------------
    # C6 FEATURES: PCI Collision
    # -------------------------------------------------------------------------
    # serving_electronic_tilt_deg, serving_total_tilt_deg, tilt_to_beamwidth_ratio
    # noncolocated_neighbor_count, colocated_neighbor_count, abnormal_path_loss
    
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])
        
        if cell:
            mech_tilt = cell.get("mechanical_downtilt") or 0.0
            elec_tilt = electronic_tilt_deg(cell.get("digital_tilt"))
            total_tilt = float(mech_tilt) + float(elec_tilt)
            
            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))
            tilt_to_beamwidth_ratio = total_tilt / vbw if vbw > 0 else 0.0
            
            features["serving_electronic_tilt_deg"] = elec_tilt
            features["serving_total_tilt_deg"] = total_tilt
            features["tilt_to_beamwidth_ratio"] = tilt_to_beamwidth_ratio
        else:
            features["serving_electronic_tilt_deg"] = None
            features["serving_total_tilt_deg"] = None
            features["tilt_to_beamwidth_ratio"] = None
    else:
        features["serving_electronic_tilt_deg"] = None
        features["serving_total_tilt_deg"] = None
        features["tilt_to_beamwidth_ratio"] = None
    
    # noncolocated_neighbor_count, colocated_neighbor_count
    neighbor_gnodebs = set()
    colocated_neighbors = []
    noncolocated_neighbors = []
    
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            nei_pci = r.get(k)
            if nei_pci is not None:
                nei_gnodeb = pci_to_gnodeb.get(nei_pci)
                if nei_gnodeb is not None:
                    neighbor_gnodebs.add(nei_gnodeb)
                    if serving_gnodeb is not None:
                        if nei_gnodeb == serving_gnodeb:
                            colocated_neighbors.append(nei_pci)
                        else:
                            noncolocated_neighbors.append(nei_pci)
    
    features["noncolocated_neighbor_count"] = len(set(noncolocated_neighbors))
    features["colocated_neighbor_count"] = len(set(colocated_neighbors))
    
    # abnormal_path_loss
    if distances and serving_rsrps and len(distances) == len(serving_rsrps):
        path_loss_deviations = []
        for i, (dist, rsrp) in enumerate(zip(distances, serving_rsrps)):
            if dist > 10:
                dist_km = dist / 1000
                if dist_km > 0:
                    expected_pl = 128.1 + 37.6 * math.log10(dist_km)
                    actual_pl = 46 - rsrp
                    deviation = abs(actual_pl - expected_pl)
                    path_loss_deviations.append(deviation)
        
        if path_loss_deviations:
            path_loss_deviation_mean = mean(path_loss_deviations)
            features["abnormal_path_loss"] = path_loss_deviation_mean > 15
        else:
            features["abnormal_path_loss"] = False
    else:
        features["abnormal_path_loss"] = False
    
    # -------------------------------------------------------------------------
    # C7 FEATURES: High Speed
    # -------------------------------------------------------------------------
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]
    
    if speeds:
        features["speed_max_kmh"] = max(speeds)
        features["speed_mean_kmh"] = mean(speeds)
        features["speed_above_40_flag"] = features["speed_max_kmh"] > 40 if features["speed_max_kmh"] else False
    else:
        features["speed_max_kmh"] = None
        features["speed_mean_kmh"] = None
        features["speed_above_40_flag"] = False
    
    # fast_low_tp_ratio, speed_tp_correlation, c7_speed_hint
    speed_tp_pairs = []
    fast_total = 0
    fast_low_tp = 0
    
    for r in drive_rows:
        spd = r.get("gps_speed_kmh")
        tp = r.get("throughput_mbps")
        if spd is None or tp is None:
            continue
        speed_tp_pairs.append((spd, tp))
        if spd >= 40:
            fast_total += 1
            if tp < 600:
                fast_low_tp += 1
    
    features["fast_low_tp_ratio"] = (fast_low_tp / fast_total) if fast_total > 0 else 0.0
    
    if speed_tp_pairs:
        speeds2 = [x for x, _ in speed_tp_pairs]
        tps2 = [y for _, y in speed_tp_pairs]
        features["speed_tp_correlation"] = _safe_corr(speeds2, tps2)
    else:
        features["speed_tp_correlation"] = None
    
    features["c7_speed_hint"] = (
        features.get("speed_above_40_flag", False) and
        features.get("fast_low_tp_ratio", 0.0) > 0.25
    )
    
    # -------------------------------------------------------------------------
    # C8 FEATURES: Resource Congestion
    # -------------------------------------------------------------------------
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]
    
    if rbs:
        features["rb_mean"] = mean(rbs)
        features["rb_min"] = min(rbs)
    else:
        features["rb_mean"] = None
        features["rb_min"] = None
    
    # high_rb_low_tp_ratio_v2, tp_drop_with_high_rb_ratio, rb_tp_correlation
    rb_tp_pairs2 = []
    high_rb_total = 0
    high_rb_low_tp = 0
    drop_total = 0
    drop_high_rb = 0
    
    for r in drive_rows:
        rb = r.get("dl_rb_num")
        tp = r.get("throughput_mbps")
        if rb is None or tp is None:
            continue
        
        rb_tp_pairs2.append((rb, tp))
        
        if rb >= 180:
            high_rb_total += 1
            if tp < 600:
                high_rb_low_tp += 1
        
        if tp < 600:
            drop_total += 1
            if rb > 180:
                drop_high_rb += 1
    
    features["high_rb_low_tp_ratio_v2"] = (high_rb_low_tp / high_rb_total) if high_rb_total > 0 else 0.0
    features["tp_drop_with_high_rb_ratio"] = (drop_high_rb / drop_total) if drop_total > 0 else 0.0
    
    if rb_tp_pairs2:
        rbs2 = [x for x, _ in rb_tp_pairs2]
        tps3 = [y for _, y in rb_tp_pairs2]
        features["rb_tp_correlation"] = _safe_corr(rbs2, tps3)
    else:
        features["rb_tp_correlation"] = None
    
    # throughput_efficiency_min
    tp_rb_pairs = [(r.get("throughput_mbps"), r.get("dl_rb_num"))
                   for r in drive_rows
                   if r.get("throughput_mbps") is not None and r.get("dl_rb_num") is not None and r.get("dl_rb_num") > 0]
    
    if tp_rb_pairs:
        efficiencies = [tp / rb for tp, rb in tp_rb_pairs]
        features["throughput_efficiency_min"] = min(efficiencies)
    else:
        features["throughput_efficiency_min"] = None
    
    return features

print("✓ RCA feature engineering ready (STREAMLINED)")
print("  Computes ONLY 48 features required by format_features_text:")
print("    C1 (6): rsrp_min_dbm, rsrp_10th_percentile, handover_rsrp_delta_mean, rsrp_mean_dbm,")
print("            sinr_degradation_db, sinr_std_when_rsrp_stable")
print("    C2 (6): dist_max_m, overshoot_flag, dist_p95_m, throughput_variance_across_cells,")
print("            handover_count, best_cell_throughput_gap")
print("    C3 (6): avg_throughput_change_after_handover, handover_tp_delta_mean, handover_improves_tp_flag,")
print("            throughput_improved_by_handover, tp_samples_below_600, tp_drop_ratio")
print("    C4 (6): noncolocated_strong_neighbor_gnodeb_count_mean, strong_neighbor_noncolocated_share,")
print("            noncolocated_strong_neighbor_gnodeb_count_max, high_interference_power_ratio_flag,")
print("            rsrp_advantage_of_best_neighbor, top1_top2_gap_db_mean")
print("    C5 (6): ping_pong_handover_count, frequent_handover_flag, ping_pong_detected,")
print("            handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean")
print("    C6 (6): serving_electronic_tilt_deg, serving_total_tilt_deg, noncolocated_neighbor_count,")
print("            abnormal_path_loss, tilt_to_beamwidth_ratio, colocated_neighbor_count")
print("    C7 (6): speed_max_kmh, c7_speed_hint, speed_above_40_flag, fast_low_tp_ratio,")
print("            speed_mean_kmh, speed_tp_correlation")
print("    C8 (6): rb_mean, high_rb_low_tp_ratio_v2, rb_min, tp_drop_with_high_rb_ratio,")
print("            rb_tp_correlation, throughput_efficiency_min")


✓ RCA feature engineering ready (STREAMLINED)
  Computes ONLY 48 features required by format_features_text:
    C1 (6): rsrp_min_dbm, rsrp_10th_percentile, handover_rsrp_delta_mean, rsrp_mean_dbm,
            sinr_degradation_db, sinr_std_when_rsrp_stable
    C2 (6): dist_max_m, overshoot_flag, dist_p95_m, throughput_variance_across_cells,
            handover_count, best_cell_throughput_gap
    C3 (6): avg_throughput_change_after_handover, handover_tp_delta_mean, handover_improves_tp_flag,
            throughput_improved_by_handover, tp_samples_below_600, tp_drop_ratio
    C4 (6): noncolocated_strong_neighbor_gnodeb_count_mean, strong_neighbor_noncolocated_share,
            noncolocated_strong_neighbor_gnodeb_count_max, high_interference_power_ratio_flag,
            rsrp_advantage_of_best_neighbor, top1_top2_gap_db_mean
    C5 (6): ping_pong_handover_count, frequent_handover_flag, ping_pong_detected,
            handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean
    C6

In [6]:
# ============================================================================
# ENHANCED QUESTION FORMATTING (Q&A Format)
# ============================================================================

def format_value(v) -> str:
    """Format value for display in prompt"""
    if v is None:
        return "N/A"
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def format_features_text(features: Dict) -> str:
    """
    Compact format focusing on top discriminating features per class.
    Optimized for token efficiency while preserving classification power.
    """
    feature_text = "\n".join([
        "**Key RCA Features:**",
        "",
        "**C1 Indicators (Excessive Downtilt):**",
        "  1. RSRP min: {0} dBm".format(format_value(features.get('rsrp_min_dbm'))),
        "  2. RSRP 10th percentile: {0} dBm".format(format_value(features.get('rsrp_10th_percentile'))),
        "  3. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  4. RSRP mean: {0} dBm".format(format_value(features.get('rsrp_mean_dbm'))),
        "  5. SINR degradation: {0} dB".format(format_value(features.get('sinr_degradation_db'))),
        "  6. SINR std when RSRP stable: {0}".format(format_value(features.get('sinr_std_when_rsrp_stable'))),
        "",
        "**C2 Indicators (Overshoot):**",
        "  1. Distance max: {0} m".format(format_value(features.get('dist_max_m'))),
        "  2. Overshoot flag: {0}".format(format_value(features.get('overshoot_flag'))),
        "  3. Distance p95: {0} m".format(format_value(features.get('dist_p95_m'))),
        "  4. TP variance across cells: {0}".format(format_value(features.get('throughput_variance_across_cells'))),
        "  5. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  6. Best cell TP gap: {0} Mbps".format(format_value(features.get('best_cell_throughput_gap'))),
        "",
        "**C3 Indicators (Wrong Cell Selection):**",
        "  1. Avg TP change after handover: {0} Mbps".format(format_value(features.get('avg_throughput_change_after_handover'))),
        "  2. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "  3. Handover improves TP flag: {0}".format(format_value(features.get('handover_improves_tp_flag'))),
        "  4. TP improved by handover: {0}".format(format_value(features.get('throughput_improved_by_handover'))),
        "  5. TP samples below 600: {0}".format(format_value(features.get('tp_samples_below_600'))),
        "  6. TP drop ratio: {0}".format(format_value(features.get('tp_drop_ratio'))),
        "",
        "**C4 Indicators (Overlapping Coverage):**",
        "  1. Non-colocated strong neighbor gNodeB count mean: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_mean'))),
        "  2. Strong neighbor non-colocated share: {0}".format(format_value(features.get('strong_neighbor_noncolocated_share'))),
        "  3. Non-colocated strong neighbor gNodeB count max: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_max'))),
        "  4. High interference power ratio flag: {0}".format(format_value(features.get('high_interference_power_ratio_flag'))),
        "  5. RSRP advantage of best neighbor: {0} dB".format(format_value(features.get('rsrp_advantage_of_best_neighbor'))),
        "  6. Top1-top2 gap dB mean: {0} dB".format(format_value(features.get('top1_top2_gap_db_mean'))),
        "",
        "**C5 Indicators (Ping-Pong Handover):**",
        "  1. Ping-pong handover count: {0}".format(format_value(features.get('ping_pong_handover_count'))),
        "  2. Frequent handover flag: {0}".format(format_value(features.get('frequent_handover_flag'))),
        "  3. Ping-pong detected: {0}".format(format_value(features.get('ping_pong_detected'))),
        "  4. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  5. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  6. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "",
        "**C6 Indicators (PCI Collision):**",
        "  1. Serving electronic tilt: {0}°".format(format_value(features.get('serving_electronic_tilt_deg'))),
        "  2. Serving total tilt: {0}°".format(format_value(features.get('serving_total_tilt_deg'))),
        "  3. Non-colocated neighbor count: {0}".format(format_value(features.get('noncolocated_neighbor_count'))),
        "  4. Abnormal path loss: {0}".format(format_value(features.get('abnormal_path_loss'))),
        "  5. Tilt to beamwidth ratio: {0}".format(format_value(features.get('tilt_to_beamwidth_ratio'))),
        "  6. Co-located neighbor count: {0}".format(format_value(features.get('colocated_neighbor_count'))),
        "",
        "**C7 Indicators (High Speed):**",
        "  1. Speed max: {0} km/h".format(format_value(features.get('speed_max_kmh'))),
        "  2. C7 speed hint: {0}".format(format_value(features.get('c7_speed_hint'))),
        "  3. Speed above 40 flag: {0}".format(format_value(features.get('speed_above_40_flag'))),
        "  4. Fast low TP ratio: {0}".format(format_value(features.get('fast_low_tp_ratio'))),
        "  5. Speed mean: {0} km/h".format(format_value(features.get('speed_mean_kmh'))),
        "  6. Speed-TP correlation: {0}".format(format_value(features.get('speed_tp_correlation'))),
        "",
        "**C8 Indicators (Resource Congestion):**",
        "  1. RB mean: {0}".format(format_value(features.get('rb_mean'))),
        "  2. High RB low TP ratio v2: {0}".format(format_value(features.get('high_rb_low_tp_ratio_v2'))),
        "  3. RB min: {0}".format(format_value(features.get('rb_min'))),
        "  4. TP drop with high RB ratio: {0}".format(format_value(features.get('tp_drop_with_high_rb_ratio'))),
        "  5. RB-TP correlation: {0}".format(format_value(features.get('rb_tp_correlation'))),
        "  6. TP efficiency min: {0}".format(format_value(features.get('throughput_efficiency_min'))),
    ])

    return feature_text

def format_enhanced_question(original_question: str, features_text: str) -> str:

    """

    Combine the original question with engineered features.print("  - format_enhanced_question(): Combines original question + features")

    This creates the enhanced question column WITHOUT the answer.print("  - format_features_text(): Formats features into readable text")

    """
    print("✓ Q&A format functions ready")

    enhanced_question = f"""
        {original_question}
        {features_text}
        """

    return enhanced_question

In [7]:
# ============================================================================
# MAIN PREPROCESSING FUNCTION (DataFrame-Based)
# ============================================================================

def preprocess_row(row: Dict) -> Optional[Dict]:
    """
    Main preprocessing pipeline that returns structured data for DataFrame.

    Returns a dict with:
    - ID: Sample identifier
    - original_question: The original question text (sanitized)
    - features_text: Formatted engineered features
    - enhanced_question: original_question + features_text
    - answer: The answer label (C1-C8)
    - features_dict: Raw feature values as dict (for analysis)

    Returns None if row cannot be processed.
    """
    # Sanitize question text
    question_text = sanitize_question_text(row["question"])
    label = str(row["answer"]).strip().upper()

    # Validate label
    if label not in CAUSE_TO_NUM:
        return None

    # Extract table sections
    drive_match = re.search(r"User plane drive test data as follows[:：]\s*", question_text)
    eng_match = re.search(r"Engeneering parameters data as follows[:：]\s*", question_text)

    if not drive_match or not eng_match or eng_match.start() <= drive_match.end():
        return None

    drive_text = question_text[drive_match.end():eng_match.start()].strip()
    eng_text = question_text[eng_match.end():].strip()

    # Parse tables
    drive_raw = parse_pipe_table(drive_text)
    eng_raw = parse_pipe_table(eng_text)

    if not drive_raw or not eng_raw:
        return None

    # Normalize with type casting (CRITICAL!)
    drive_rows = normalize_rows(drive_raw, DRIVE_MAP)
    eng_rows = normalize_rows(eng_raw, ENG_MAP)

    # Compute RCA features
    features = compute_rca_features(drive_rows, eng_rows)

    # Format features as text
    features_text = format_features_text(features)

    # Enhanced question = original + features (NO answer)
    enhanced_question = format_enhanced_question(question_text, features_text)

    # Return structured data for DataFrame
    return {
        "ID": row["ID"],
        "original_question": question_text,
        "features_text": features_text,
        "enhanced_question": enhanced_question,
        "answer": label,
        "features_dict": features  # Keep raw dict for analysis
    }

print("✓ DataFrame-based preprocessing pipeline ready")
print("  Returns dict with columns:")
print("    - ID: Sample identifier")
print("    - original_question: Original question text")
print("    - features_text: Formatted engineered features")
print("    - enhanced_question: Question + features combined")
print("    - answer: Root cause label (C1-C8)")
print("    - features_dict: Raw feature values (for analysis)")


✓ DataFrame-based preprocessing pipeline ready
  Returns dict with columns:
    - ID: Sample identifier
    - original_question: Original question text
    - features_text: Formatted engineered features
    - enhanced_question: Question + features combined
    - answer: Root cause label (C1-C8)
    - features_dict: Raw feature values (for analysis)


## 🚀 Apply Enhanced Preprocessing

Now let's process the training data with the improved pipeline that includes:
- Proper type casting (no more string/number bugs)
- RCA-friendly feature engineering
- Enhanced chat formatting

In [8]:
df_train.head()

,ID,question,answer
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test use...,C2
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test use...,C1
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test use...,C2
3,ID_D6C9N2X295,Analyze the 5G wireless network drive-test use...,C2
4,ID_8JC15PNP3Q,Analyze the 5G wireless network drive-test use...,C5


In [9]:
len(df_train.to_dict(orient='records'))

2400

In [10]:
# Process all training samples into DataFrame format
print("=" * 70)
print("PROCESSING TRAINING DATA INTO DATAFRAME FORMAT")
print("=" * 70)
print(f"Total samples to process: {len(df_train):,}\n")

processed_records = []
failed_count = 0

for idx, row_dict in enumerate(df_train.to_dict(orient="records")):
    if idx % 10000 == 0 and idx > 0:
        print(f"  Processed {idx:,} samples... ({len(processed_records):,} successful)")

    result = preprocess_row(row_dict)

    if result is not None:
        processed_records.append(result)
    else:
        failed_count += 1

print(f"\n{'='*70}")
print(f"✓ Processing complete!")
print(f"  - Successfully processed: {len(processed_records):,}")
print(f"  - Failed: {failed_count:,}")
print(f"  - Success rate: {100 * len(processed_records) / len(df_train):.1f}%")
print(f"{'='*70}\n")

# Create DataFrame with structured columns
df_processed = pd.DataFrame(processed_records)

print("✓ Created DataFrame with columns:")
for col in df_processed.columns:
    if col != 'features_dict':  # Don't show dict column details
        print(f"  - {col}")
print(f"\nDataFrame shape: {df_processed.shape}")
print(f"Memory usage: {df_processed.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


PROCESSING TRAINING DATA INTO DATAFRAME FORMAT
Total samples to process: 2,400

✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format function

In [12]:
# TEST SET

# test = pd.read_csv('/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/phase_1_test.csv')
test = pd.read_csv('data/phase_1_test.csv')
test["answer"] = "C5"

# Process all training samples into DataFrame format
print("=" * 70)
print("PROCESSING TEST DATA INTO DATAFRAME FORMAT")
print("=" * 70)
print(f"Total samples to process: {len(test):,}\n")

test_processed_records = []
test_failed_count = 0

for idx, row_dict in enumerate(test.to_dict(orient="records")):
    if idx % 10000 == 0 and idx > 0:
        print(f"  Processed {idx:,} samples... ({len(test_processed_records):,} successful)")

    result = preprocess_row(row_dict)

    if result is not None:
        test_processed_records.append(result)
    else:
        test_failed_count += 1

print(f"\n{'='*70}")
print(f"✓ Processing complete!")
print(f"  - Successfully processed: {len(test_processed_records):,}")
print(f"  - Failed: {test_failed_count:,}")
print(f"  - Success rate: {100 * len(test_processed_records) / len(test):.1f}%")
print(f"{'='*70}\n")

# Create DataFrame with structured columns
test_df_processed = pd.DataFrame(test_processed_records)

print("✓ Created DataFrame with columns:")
for col in test_df_processed.columns:
    if col != 'features_dict':  # Don't show dict column details
        print(f"  - {col}")
print(f"\nDataFrame shape: {test_df_processed.shape}")
print(f"Memory usage: {test_df_processed.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


PROCESSING TEST DATA INTO DATAFRAME FORMAT
Total samples to process: 864

✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions ready
✓ Q&A format functions read

In [17]:
# ============================================================================
# OPTIMIZED PROMPT FORMAT FOR QWEN 7B INSTRUCT - RAN SPECIALIST
# ============================================================================

"""
KEY PRINCIPLES FOR MAXIMUM ACCURACY:

1. STRUCTURED REASONING: Train the model to explain its reasoning before answering
2. EXPLICIT DISCRIMINATORS:
 Include clear decision rules for each class
3. PROGRESSIVE COMPLEXITY: Start with easy cases, add edge cases
4. CHAIN-OF-THOUGHT: Force the model to think step-by-step
5. NEGATIVE EXAMPLES: Include what each class is NOT
"""

def format_optimal_training_prompt(row: Dict) -> Dict:
    """
    Creates optimal training format for Qwen 7B with:
    - Explicit domain knowledge
    - Step-by-step reasoning
    - Clear decision rules
    - Pattern recognition guidance
    """

    # Extract the engineered features
    features = row['features_dict']
    original_q = row['original_question']
    answer = row['answer']

    # Build enhanced system prompt with domain expertise
    EXPERT_SYSTEM = """You are an expert 5G RAN (Radio Access Network) optimization specialist with deep knowledge of root cause analysis.

Your expertise includes:
- Coverage issues (weak signal, excessive tilt, path loss)
- Cell selection problems (wrong cell serving, penetration loss)
- Interference patterns (overlapping coverage, co-channel interference)
- Mobility issues (handover problems, ping-pong handovers)
- PCI collision and confusion
- High-speed performance degradation
- Resource congestion and scheduling inefficiency

ANALYSIS FRAMEWORK:
1. Examine RSRP/SINR patterns (signal strength vs quality)
2. Analyze neighbor cell relationships (co-located vs non-co-located)
3. Check handover behavior and cell selection
4. Evaluate throughput efficiency and resource utilization
5. Consider speed and mobility factors
6. Look for systematic patterns vs sporadic issues

ROOT CAUSE CLASSES:
- C1: Excessive Downtilt (geometry issue, weak far coverage)
- C2: Overshoot (serving cell too far, distance >1km)
- C3: Wrong Cell Selection (better neighbor available, penetration loss)
- C4: Overlapping Coverage (multiple non-co-located sites, interference)
- C5: Ping-Pong Handover (frequent back-and-forth switching)
- C6: PCI Collision/Confusion (PCI reuse, mod 30 collision)
- C7: High Speed Impact (mobility degradation >40 km/h)
- C8: Resource Congestion (high RB allocation but low throughput)

Provide your analysis step-by-step, then give final answer as \\boxed{{n}} where n=1..8."""

    # Build reasoning-enhanced user prompt
    USER_PROMPT = f"""{original_q}

{row['features_text']}

**Analysis Instructions:**
Think step-by-step:

1. **Signal Quality Assessment:**
   - What are the RSRP and SINR levels?
   - Are they correlated (both low = C1/C3) or decoupled (good RSRP + poor SINR = C4)?

2. **Neighbor Analysis:**
   - How many strong neighbors exist?
   - Are they from the same gNodeB (co-located) or different sites (non-co-located)?
   - Is there one dominant neighbor (C3) or multiple equal neighbors (C4)?

3. **Mobility & Handover:**
   - How many handovers occurred?
   - Do handovers improve throughput (C3/C5) or not?
   - Is there ping-pong behavior (back-and-forth)?

4. **Efficiency Metrics:**
   - Is RB allocation high but throughput low (C8)?
   - Does speed correlate with poor performance (C7)?
   - Is there abnormal path loss or tilt issues (C1)?

5. **Distance & Coverage:**
   - Is the UE very far from the cell (C2)?
   - Are there coverage holes or weak spots (C1)?

Based on this systematic analysis, identify the PRIMARY root cause."""

    # Build reasoning-based assistant response
    # This is the KEY: train the model to reason before answering
    ASSISTANT_REASONING = generate_reasoning_for_class(answer, features)

    ASSISTANT_RESPONSE = f"""{ASSISTANT_REASONING}

**Final Answer:** \\boxed{{{answer[1]}}}"""  # Extract number from "C1" -> "1"

    return {
        "messages": [
            {"role": "system", "content": EXPERT_SYSTEM},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant", "content": ASSISTANT_RESPONSE}
        ]
    }


def generate_reasoning_for_class(answer: str, features: Dict) -> str:
    """
    Generate class-specific reasoning based on the actual features.
    This teaches the model the DECISION PROCESS, not just the answer.
    """

    # Extract key metrics
    rsrp_mean = features.get('rsrp_mean_dbm', 'N/A')
    sinr_mean = features.get('sinr_mean_db', 'N/A')
    dist_p95 = features.get('dist_p95_m', 'N/A')
    ho_count = features.get('handover_count', 0)
    speed_max = features.get('speed_max_kmh', 'N/A')
    rb_mean = features.get('rb_mean', 'N/A')
    tp_drop = features.get('tp_drop_ratio', 'N/A')

    # Class-specific reasoning templates
    reasoning_templates = {
        "C1": f"""**Step-by-step Analysis:**

1. **Signal Quality:** RSRP mean = {rsrp_mean} dBm (weak), SINR = {sinr_mean} dB
   - Weak signal especially at cell edge (RSRP min: {features.get('rsrp_min_dbm', 'N/A')} dBm)
   - RSRP 10th percentile: {features.get('rsrp_10th_percentile', 'N/A')} dBm indicates poor far-end coverage

2. **Geometry Issue:** Tilt = {features.get('serving_total_tilt_deg', 'N/A')}°, Distance p95 = {dist_p95}m
   - Excessive tilt causes main beam to undershoot distant UEs
   - SINR degradation: {features.get('sinr_degradation_db', 'N/A')} dB when RSRP stable

3. **Not C3/C4:** No significantly better neighbor available
   - Best neighbor advantage: {features.get('rsrp_advantage_of_best_neighbor', 'N/A')} dB (not dominant)

4. **Not C8:** Issue is coverage, not congestion (RB allocation normal)

**Root Cause Identification:** EXCESSIVE DOWNTILT (C1)
The weak far-end coverage with abnormal path loss and high tilt-to-beamwidth ratio indicates geometry/coverage issue.""",

        "C2": f"""**Step-by-step Analysis:**

1. **Distance Assessment:** Distance max = {features.get('dist_max_m', 'N/A')}m, p95 = {dist_p95}m
   - UE is very far from serving cell (>1000m threshold for overshoot)
   - Overshoot flag: {features.get('overshoot_flag', 'N/A')}

2. **Alternative Cells:** Handovers = {ho_count}, TP variance across cells = {features.get('throughput_variance_across_cells', 'N/A')} Mbps
   - Multiple cells visible but wrong cell is serving
   - Best cell TP gap: {features.get('best_cell_throughput_gap', 'N/A')} Mbps

3. **Not C1:** Not about tilt/geometry but about serving area boundaries

4. **Not C3:** Issue is distance, not indoor penetration or specific cell selection

**Root Cause Identification:** OVERSHOOT (C2)
The UE is being served by a cell that is too far away, causing coverage boundary issues.""",

        "C3": f"""**Step-by-step Analysis:**

1. **Handover Impact:** Avg TP change after handover = {features.get('avg_throughput_change_after_handover', 'N/A')} Mbps
   - Handover TP delta mean: {features.get('handover_tp_delta_mean', 'N/A')} Mbps (significant improvement)
   - Handover improves TP flag: {features.get('handover_improves_tp_flag', 'N/A')}

2. **Cell Selection:** Throughput improved by handover: {features.get('throughput_improved_by_handover', 'N/A')}
   - TP samples below 600: {features.get('tp_samples_below_600', 0)} samples
   - TP drop ratio: {tp_drop}

3. **Signal Pattern:** Both RSRP and SINR are degraded (correlated)
   - Not decoupled like C4 (interference)
   - Indicates penetration loss or wrong cell serving

4. **Neighbor Dominance:** Single dominant neighbor available
   - Not symmetric interference pattern (C4)

**Root Cause Identification:** WRONG CELL SELECTION (C3)
The UE should be served by a different cell - handovers consistently improve throughput.""",

        "C4": f"""**Step-by-step Analysis:**

1. **Multi-site Interference:** Non-colocated strong neighbor gNodeB count = {features.get('noncolocated_strong_neighbor_gnodeb_count_mean', 'N/A')}
   - Strong neighbor non-colocated share: {features.get('strong_neighbor_noncolocated_share', 'N/A')}
   - Multiple non-colocated sites causing interference

2. **Signal Decoupling:** RSRP = {rsrp_mean} dBm (good), SINR = {sinr_mean} dB (poor)
   - High interference power ratio flag: {features.get('high_interference_power_ratio_flag', 'N/A')}
   - Good signal strength but poor quality = interference

3. **Symmetry Pattern:** Top1-top2 gap = {features.get('top1_top2_gap_db_mean', 'N/A')} dB (small gap)
   - Multiple neighbors equally strong (no clear winner)
   - RSRP advantage of best neighbor: {features.get('rsrp_advantage_of_best_neighbor', 'N/A')} dB

4. **Not C3:** Not about one better cell, but multiple competing cells

**Root Cause Identification:** OVERLAPPING COVERAGE (C4)
Multiple non-colocated sites with symmetric interference causing poor SINR despite good RSRP.""",

        "C5": f"""**Step-by-step Analysis:**

1. **Ping-Pong Detection:** Ping-pong handover count = {features.get('ping_pong_handover_count', 0)}
   - Frequent handover flag: {features.get('frequent_handover_flag', 'N/A')}
   - Ping-pong detected: {features.get('ping_pong_detected', 'N/A')}

2. **Handover Frequency:** Total handovers = {ho_count} (≥3 is frequent)
   - Handover RSRP delta mean: {features.get('handover_rsrp_delta_mean', 'N/A')} dB
   - Handover TP delta mean: {features.get('handover_tp_delta_mean', 'N/A')} Mbps

3. **Pattern:** Back-and-forth switching between cells
   - Not unidirectional mobility (C7)
   - Not one-time cell selection issue (C3)

4. **Impact:** Excessive handover signaling and service disruption

**Root Cause Identification:** PING-PONG HANDOVER (C5)
Frequent back-and-forth handovers indicate hysteresis or handover parameter issues.""",

        "C6": f"""**Step-by-step Analysis:**

1. **Configuration Issues:** Serving electronic tilt = {features.get('serving_electronic_tilt_deg', 'N/A')}°
   - Serving total tilt: {features.get('serving_total_tilt_deg', 'N/A')}°
   - Non-colocated neighbor count: {features.get('noncolocated_neighbor_count', 'N/A')}

2. **Path Loss Anomaly:** Abnormal path loss: {features.get('abnormal_path_loss', 'N/A')}
   - Tilt to beamwidth ratio: {features.get('tilt_to_beamwidth_ratio', 'N/A')}
   - Co-located neighbor count: {features.get('colocated_neighbor_count', 'N/A')}

3. **PCI Pattern:** Configuration or PCI-related issues
   - Not typical interference (C4) or coverage (C1) pattern
   - Specific to cell configuration

4. **Systematic Issue:** Affects multiple neighbors in specific way

**Root Cause Identification:** PCI COLLISION/CONFIGURATION (C6)
PCI confusion or systematic configuration issue affecting cell identification.""",

        "C7": f"""**Step-by-step Analysis:**

1. **Speed Impact:** Speed max = {speed_max} km/h (>40 km/h threshold)
   - C7 speed hint: {features.get('c7_speed_hint', 'N/A')}
   - Speed above 40 flag: {features.get('speed_above_40_flag', 'N/A')}

2. **Mobility Degradation:** Fast low TP ratio = {features.get('fast_low_tp_ratio', 'N/A')}
   - Speed mean: {features.get('speed_mean_kmh', 'N/A')} km/h
   - Speed-TP correlation: {features.get('speed_tp_correlation', 'N/A')}

3. **Pattern:** Performance degrades specifically at high speeds
   - Not static coverage issue (C1/C2)
   - Not interference pattern (C4)

4. **Cause:** Doppler, tracking issues, or handover delays at high speed

**Root Cause Identification:** HIGH SPEED IMPACT (C7)
Performance degradation correlated with high mobility (>40 km/h).""",

        "C8": f"""**Step-by-step Analysis:**

1. **Resource Allocation:** RB mean = {rb_mean} (high allocation)
   - High RB low TP ratio: {features.get('high_rb_low_tp_ratio_v2', 'N/A')}
   - RB min: {features.get('rb_min', 'N/A')}

2. **Efficiency Issue:** TP drop with high RB ratio = {features.get('tp_drop_with_high_rb_ratio', 'N/A')}
   - RB-TP correlation: {features.get('rb_tp_correlation', 'N/A')} (weak/negative)
   - TP efficiency min: {features.get('throughput_efficiency_min', 'N/A')}

3. **Pattern:** High resource allocation but low throughput realization
   - Not coverage issue (RSRP adequate)
   - Not interference (SINR acceptable)

4. **Cause:** Network congestion, poor scheduling, or backhaul limitation

**Root Cause Identification:** RESOURCE CONGESTION (C8)
High RB allocation with poor throughput efficiency indicates congestion or scheduling issues."""
    }

    return reasoning_templates.get(answer, "**Analysis:** Based on the features provided...")


print("✓ Optimal training format for Qwen 7B created")
print("  Key improvements:")
print("    - Structured reasoning before answer")
print("    - Class-specific decision logic")
print("    - Feature-based explanations")
print("    - Teaches WHY, not just WHAT")

✓ Optimal training format for Qwen 7B created
  Key improvements:
    - Structured reasoning before answer
    - Class-specific decision logic
    - Feature-based explanations
    - Teaches WHY, not just WHAT


In [18]:
# ============================================================================
# ADVANCED TRAINING STRATEGIES FOR PERFECT ACCURACY
# ============================================================================

"""
STRATEGY 1: CURRICULUM LEARNING
Start with easy, clear-cut cases, then add ambiguous cases
"""

def classify_sample_difficulty(features: Dict) -> str:
    """
    Classify training samples by difficulty for curriculum learning
    """
    # Easy cases: Clear discriminators active
    c1_clear = features.get('c1_signature', False)
    c3_clear = features.get('c3_signature', False)
    c4_clear = features.get('c4_signature', False)

    ping_pong = features.get('ping_pong_handover_count', 0) >= 2
    high_speed = features.get('speed_above_40_flag', False)
    overshoot = features.get('overshoot_flag', False)
    high_rb_low_tp = features.get('high_rb_low_tp_ratio_v2', 0) > 0.3

    # Count how many clear signals exist
    clear_signals = sum([
        c1_clear, c3_clear, c4_clear, ping_pong,
        high_speed, overshoot, high_rb_low_tp
    ])

    if clear_signals >= 2:
        return "EASY"  # Multiple clear indicators
    elif clear_signals == 1:
        return "MEDIUM"  # One clear indicator
    else:
        return "HARD"  # Ambiguous case


"""
STRATEGY 2: DATA AUGMENTATION
Add paraphrased reasoning paths for same data
"""

def create_alternative_reasoning(answer: str, features: Dict, variant: int = 1) -> str:
    """
    Generate alternative reasoning paths for the same answer.
    This helps the model learn multiple valid approaches.
    """
    if variant == 1:
        # Elimination approach
        return f"""**Elimination Analysis:**

Let me systematically eliminate each possibility:

❌ C1 (Excessive Downtilt): {"✓ MATCH" if answer == "C1" else "No - tilt/geometry patterns don't match"}
❌ C2 (Overshoot): {"✓ MATCH" if answer == "C2" else "No - distance within normal range"}
❌ C3 (Wrong Cell): {"✓ MATCH" if answer == "C3" else "No - handover doesn't significantly improve throughput"}
❌ C4 (Overlapping Coverage): {"✓ MATCH" if answer == "C4" else "No - not multiple strong non-colocated neighbors"}
❌ C5 (Ping-Pong): {"✓ MATCH" if answer == "C5" else "No - no back-and-forth handover pattern"}
❌ C6 (PCI Collision): {"✓ MATCH" if answer == "C6" else "No - no PCI confusion indicators"}
❌ C7 (High Speed): {"✓ MATCH" if answer == "C7" else "No - speed not a factor"}
❌ C8 (Congestion): {"✓ MATCH" if answer == "C8" else "No - RB allocation and efficiency normal"}

**Conclusion:** {answer}"""

    elif variant == 2:
        # Decision tree approach
        return f"""**Decision Tree Analysis:**

1. Is RSRP adequate (>-90 dBm)?
   {"→ Yes, check SINR" if features.get('rsrp_mean_dbm', -999) > -90 else "→ No, likely C1/C2/C3"}

2. Is SINR adequate (>10 dB)?
   {"→ No, likely C4 (good RSRP + poor SINR = interference)" if features.get('sinr_mean_db', 999) < 10 else "→ Yes, check other factors"}

3. Multiple strong neighbors from different sites?
   {"→ Yes, confirms C4" if features.get('noncolocated_strong_neighbor_gnodeb_count_mean', 0) > 1.5 else "→ No, check mobility"}

4. High handover count (≥3)?
   {"→ Yes, check if ping-pong (C5)" if features.get('handover_count', 0) >= 3 else "→ No, check speed"}

5. Speed >40 km/h with poor performance?
   {"→ Yes, likely C7" if features.get('speed_above_40_flag', False) else "→ No, check resources"}

6. High RB allocation with low throughput?
   {"→ Yes, likely C8" if features.get('high_rb_low_tp_ratio_v2', 0) > 0.3 else "→ Check distance"}

**Result:** {answer}"""

    return ""


"""
STRATEGY 3: CONTRASTIVE LEARNING
Explicitly teach what each class is NOT
"""

def add_contrastive_examples(answer: str, features: Dict) -> str:
    """
    Add explicit negative examples to teach class boundaries
    """
    contrasts = {
        "C1": "NOT C3 (no better neighbor), NOT C4 (poor SINR due to weak signal, not interference)",
        "C2": "NOT C1 (issue is distance, not geometry), NOT C3 (not about cell selection)",
        "C3": "NOT C1 (RSRP adequate near cell), NOT C4 (one dominant neighbor, not multiple)",
        "C4": "NOT C1 (RSRP good), NOT C3 (multiple competitors, no single winner)",
        "C5": "NOT C7 (repeated switching, not speed-related), NOT C3 (back-and-forth, not one-way)",
        "C6": "Configuration/PCI specific, NOT C4 (not interference pattern)",
        "C7": "NOT C5 (speed-specific, not handover frequency), NOT C8 (mobility, not congestion)",
        "C8": "NOT C4 (congestion, not interference), NOT C1 (resources, not coverage)"
    }

    return f"\n**Critical Distinctions:** {contrasts.get(answer, '')}"


"""
STRATEGY 4: CONFIDENCE CALIBRATION
Teach the model to recognize ambiguous cases
"""

def add_confidence_level(answer: str, features: Dict) -> str:
    """
    Add confidence assessment to teach model uncertainty awareness
    """
    difficulty = classify_sample_difficulty(features)

    confidence_map = {
        "EASY": "HIGH (multiple clear discriminators)",
        "MEDIUM": "MEDIUM (one primary discriminator)",
        "HARD": "MEDIUM-LOW (requires careful analysis of subtle patterns)"
    }

    return f"\n**Confidence Level:** {confidence_map.get(difficulty, 'MEDIUM')}"


print("✓ Advanced training strategies loaded")
print("  Strategies available:")
print("    1. Curriculum learning (easy → hard)")
print("    2. Data augmentation (alternative reasoning)")
print("    3. Contrastive learning (what it's NOT)")
print("    4. Confidence calibration (uncertainty awareness)")

✓ Advanced training strategies loaded
  Strategies available:
    1. Curriculum learning (easy → hard)
    2. Data augmentation (alternative reasoning)
    3. Contrastive learning (what it's NOT)
    4. Confidence calibration (uncertainty awareness)


In [19]:
# ============================================================================
# COMPLETE TRAINING DATA GENERATION WITH ALL STRATEGIES
# ============================================================================

def generate_optimized_training_data(df_processed: pd.DataFrame,
                                     use_curriculum: bool = True,
                                     use_augmentation: bool = True,
                                     use_contrastive: bool = True,
                                     augmentation_factor: float = 1.5) -> list:
    """
    Generate optimized training data with all enhancement strategies.

    Args:
        df_processed: DataFrame with processed samples
        use_curriculum: Sort by difficulty (easy first)
        use_augmentation: Add alternative reasoning variants
        use_contrastive: Add negative examples
        augmentation_factor: How many augmented versions per sample (1.0 = original only)

    Returns:
        List of training examples in chat format
    """
    training_data = []

    print("Generating optimized training data...")
    print(f"  - Curriculum learning: {use_curriculum}")
    print(f"  - Data augmentation: {use_augmentation} (factor: {augmentation_factor})")
    print(f"  - Contrastive learning: {use_contrastive}")
    print()

    # Add difficulty classification
    if use_curriculum:
        df_processed['difficulty'] = df_processed['features_dict'].apply(classify_sample_difficulty)
        # Sort: EASY → MEDIUM → HARD
        difficulty_order = {'EASY': 0, 'MEDIUM': 1, 'HARD': 2}
        df_processed['difficulty_rank'] = df_processed['difficulty'].map(difficulty_order)
        df_processed = df_processed.sort_values('difficulty_rank')

    # Generate training examples
    for idx, row in df_processed.iterrows():
        features = row['features_dict']
        answer = row['answer']

        # Original reasoning
        base_reasoning = generate_reasoning_for_class(answer, features)

        # Add contrastive learning
        if use_contrastive:
            base_reasoning += add_contrastive_examples(answer, features)

        # Add confidence calibration
        base_reasoning += add_confidence_level(answer, features)

        # Create base example
        example = create_training_example(
            system_prompt=get_expert_system_prompt(),
            user_prompt=row['enhanced_question'],
            assistant_reasoning=base_reasoning,
            answer=answer
        )
        training_data.append(example)

        # Data augmentation: add variant reasoning paths
        if use_augmentation and augmentation_factor > 1.0:
            # Add 1-2 variants based on augmentation factor
            num_variants = int((augmentation_factor - 1.0) * 2)
            for variant in range(1, num_variants + 1):
                alt_reasoning = create_alternative_reasoning(answer, features, variant)
                if use_contrastive:
                    alt_reasoning += add_contrastive_examples(answer, features)
                alt_reasoning += add_confidence_level(answer, features)

                variant_example = create_training_example(
                    system_prompt=get_expert_system_prompt(),
                    user_prompt=row['enhanced_question'],
                    assistant_reasoning=alt_reasoning,
                    answer=answer
                )
                training_data.append(variant_example)

    print(f"\n✓ Generated {len(training_data):,} training examples")
    if use_augmentation:
        print(f"  - Original samples: {len(df_processed):,}")
        print(f"  - Augmented samples: {len(training_data) - len(df_processed):,}")

    return training_data


def create_training_example(system_prompt: str, user_prompt: str,
                           assistant_reasoning: str, answer: str) -> Dict:
    """Helper to create a single training example"""
    answer_num = answer[1]  # Extract "1" from "C1"

    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": f"{assistant_reasoning}\n\n**Final Answer:** \\boxed{{{answer_num}}}"}
        ]
    }


def get_expert_system_prompt() -> str:
    """Returns the expert system prompt"""
    return """You are an expert 5G RAN (Radio Access Network) optimization specialist with deep knowledge of root cause analysis.

Your expertise includes:
- Coverage issues (weak signal, excessive tilt, path loss)
- Cell selection problems (wrong cell serving, penetration loss)
- Interference patterns (overlapping coverage, co-channel interference)
- Mobility issues (handover problems, ping-pong handovers)
- PCI collision and confusion
- High-speed performance degradation
- Resource congestion and scheduling inefficiency

ANALYSIS FRAMEWORK:
1. Examine RSRP/SINR patterns (signal strength vs quality)
2. Analyze neighbor cell relationships (co-located vs non-co-located)
3. Check handover behavior and cell selection
4. Evaluate throughput efficiency and resource utilization
5. Consider speed and mobility factors
6. Look for systematic patterns vs sporadic issues

ROOT CAUSE CLASSES:
- C1: Excessive Downtilt (geometry issue, weak far coverage)
- C2: Overshoot (serving cell too far, distance >1km)
- C3: Wrong Cell Selection (better neighbor available, penetration loss)
- C4: Overlapping Coverage (multiple non-co-located sites, interference)
- C5: Ping-Pong Handover (frequent back-and-forth switching)
- C6: PCI Collision/Confusion (PCI reuse, mod 30 collision)
- C7: High Speed Impact (mobility degradation >40 km/h)
- C8: Resource Congestion (high RB allocation but low throughput)

Analyze step-by-step, explain your reasoning, then provide final answer as \\boxed{{n}} where n=1..8."""


print("✓ Complete training data generation pipeline ready")
print("\nExample usage:")
print("  optimized_data = generate_optimized_training_data(")
print("      df_processed,")
print("      use_curriculum=True,")
print("      use_augmentation=True,")
print("      use_contrastive=True,")
print("      augmentation_factor=1.5  # 50% more samples with variants")
print("  )")

✓ Complete training data generation pipeline ready

Example usage:
  optimized_data = generate_optimized_training_data(
      df_processed,
      use_curriculum=True,
      use_augmentation=True,
      use_contrastive=True,
      augmentation_factor=1.5  # 50% more samples with variants
  )


In [20]:
# ============================================================================
# EXAMPLE: GENERATE AND PREVIEW OPTIMIZED TRAINING DATA
# ============================================================================

# Generate sample to see the format
print("="*80)
print("GENERATING SAMPLE OPTIMIZED TRAINING DATA")
print("="*80)

# Take a few samples from each class
sample_rows = []
for class_label in ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']:
    class_samples = df_processed[df_processed['answer'] == class_label].head(1)
    if len(class_samples) > 0:
        sample_rows.append(class_samples.iloc[0])

sample_df = pd.DataFrame(sample_rows)

# Show original vs optimized format comparison
print("\n" + "="*80)
print("COMPARING FORMATS")
print("="*80)

if len(sample_df) > 0:
    sample_row = sample_df.iloc[0]

    print("\n### ORIGINAL FORMAT (Current) ###")
    print("-" * 80)
    original = {
        "messages": [
            {"role": "system", "content": "You are a 5G RAN optimization assistant..."},
            {"role": "user", "content": sample_row['enhanced_question'][:500] + "..."},
            {"role": "assistant", "content": f"\\boxed{{{sample_row['answer'][1]}}}"}
        ]
    }
    print(f"System: {original['messages'][0]['content'][:100]}...")
    print(f"User: {original['messages'][1]['content'][:200]}...")
    print(f"Assistant: {original['messages'][2]['content']}")
    print(f"Total tokens (approx): ~1000")

    print("\n\n### OPTIMIZED FORMAT (With Reasoning) ###")
    print("-" * 80)
    features = sample_row['features_dict']
    reasoning = generate_reasoning_for_class(sample_row['answer'], features)
    reasoning += add_contrastive_examples(sample_row['answer'], features)
    reasoning += add_confidence_level(sample_row['answer'], features)

    optimized = {
        "messages": [
            {"role": "system", "content": get_expert_system_prompt()},
            {"role": "user", "content": sample_row['enhanced_question'][:]},
            {"role": "assistant", "content": f"{reasoning[:]}\n\n**Final Answer:** \\boxed{{{sample_row['answer'][1]}}}"}
        ]
    }
    print(f"System: {optimized['messages'][0]['content'][:]}")
    print(f"User: {optimized['messages'][1]['content'][:]}")
    print(f"Assistant (with reasoning): {optimized['messages'][2]['content'][:]}")
    print(f"Total tokens (approx): ~2500-3000")

    print("\n" + "="*80)
    print("KEY DIFFERENCES")
    print("="*80)
    print("✓ Explicit domain expertise in system prompt")
    print("✓ Step-by-step reasoning before answer")
    print("✓ Feature-based justification")
    print("✓ Contrastive examples (what it's NOT)")
    print("✓ Confidence calibration")
    print("✓ Teaches decision process, not just memorization")
    print("="*80)

GENERATING SAMPLE OPTIMIZED TRAINING DATA

COMPARING FORMATS

### ORIGINAL FORMAT (Current) ###
--------------------------------------------------------------------------------
System: You are a 5G RAN optimization assistant......
User: 
        Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the followi...
Assistant: \boxed{1}
Total tokens (approx): ~1000


### OPTIMIZED FORMAT (With Reasoning) ###
--------------------------------------------------------------------------------
System: You are an expert 5G RAN (Radio Access Network) optimization specialist with deep knowledge of root cause analysis.

Your expertise includes:
- Coverage issues (weak signal, excessive tilt, path loss)
- Cell selection problems (wrong cell serving, penetration loss)
- Interference patterns (overlapping coverage, co-channel interference)
- Mobility issues (handover problems, 

In [21]:
optimized

{'messages': [{'role': 'system',
   'content': 'You are an expert 5G RAN (Radio Access Network) optimization specialist with deep knowledge of root cause analysis.\n\nYour expertise includes:\n- Coverage issues (weak signal, excessive tilt, path loss)\n- Cell selection problems (wrong cell serving, penetration loss)\n- Interference patterns (overlapping coverage, co-channel interference)\n- Mobility issues (handover problems, ping-pong handovers)\n- PCI collision and confusion\n- High-speed performance degradation\n- Resource congestion and scheduling inefficiency\n\nANALYSIS FRAMEWORK:\n1. Examine RSRP/SINR patterns (signal strength vs quality)\n2. Analyze neighbor cell relationships (co-located vs non-co-located)\n3. Check handover behavior and cell selection\n4. Evaluate throughput efficiency and resource utilization\n5. Consider speed and mobility factors\n6. Look for systematic patterns vs sporadic issues\n\nROOT CAUSE CLASSES:\n- C1: Excessive Downtilt (geometry issue, weak far c